# Notebook 05: Bronze-to-Silver Transformation

## Objective

This notebook transforms the PaySim Bronze dataset into a validated and
standardized Silver dataset.

The Silver layer will:

- preserve source lineage metadata;
- standardize column names;
- derive transaction day and hour;
- classify destination accounts as customers or merchants;
- create transaction-level business indicators;
- apply hard validation rules;
- apply soft warning rules;
- separate valid records from quarantine records;
- assign validation reasons;
- reconcile Bronze, Silver, and quarantine record counts.

The Silver layer contains clean, analytics-ready transactions while preserving
traceability back to the original source record.

In [2]:
import os
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [3]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python executable:", sys.executable)

Python executable: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe


In [4]:
spark = (
    SparkSession.builder
    .appName("PaySimBronzeToSilver")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark master: local[4]


In [5]:
PROJECT_ROOT = Path(
    r"C:\Projects\paysim-financial-data-pipeline"
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

AUDIT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "pipeline_audit"
)

QUALITY_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "silver_quality"
)

AUDIT_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

QUALITY_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw source:", RAW_DATA_PATH)
print("Source exists:", RAW_DATA_PATH.exists())

Raw source: C:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
Source exists: True


In [6]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Source file not found: {RAW_DATA_PATH}"
    )

In [7]:
transaction_schema = StructType(
    [
        StructField("step", IntegerType(), nullable=True),
        StructField("type", StringType(), nullable=True),
        StructField("amount", DoubleType(), nullable=True),
        StructField("nameOrig", StringType(), nullable=True),
        StructField("oldbalanceOrg", DoubleType(), nullable=True),
        StructField("newbalanceOrig", DoubleType(), nullable=True),
        StructField("nameDest", StringType(), nullable=True),
        StructField("oldbalanceDest", DoubleType(), nullable=True),
        StructField("newbalanceDest", DoubleType(), nullable=True),
        StructField("isFraud", IntegerType(), nullable=True),
        StructField("isFlaggedFraud", IntegerType(), nullable=True),
    ]
)

In [8]:
pipeline_run_id = str(uuid.uuid4())

pipeline_start_timestamp = datetime.now(
    timezone.utc
)

source_filename = RAW_DATA_PATH.name

print("Pipeline run ID:", pipeline_run_id)
print("Pipeline start:", pipeline_start_timestamp)
print("Source file:", source_filename)

Pipeline run ID: 25e09ba1-b996-4bef-9d75-2b409e67afdd
Pipeline start: 2026-07-24 18:44:16.915965+00:00
Source file: PS_20174392719_1491204439457_log.csv


In [9]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(transaction_schema)
    .csv(str(RAW_DATA_PATH))
)

In [10]:
raw_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [11]:
SOURCE_COLUMNS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud",
]

In [12]:
record_hash_expression = F.sha2(
    F.concat_ws(
        "||",
        *[
            F.coalesce(
                F.col(column).cast("string"),
                F.lit("<NULL>"),
            )
            for column in SOURCE_COLUMNS
        ],
    ),
    256,
)

In [13]:
bronze_df = (
    raw_df
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id),
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date(F.current_timestamp()),
    )
    .withColumn(
        "_source_file",
        F.lit(source_filename),
    )
    .withColumn(
        "_source_file_path",
        F.lit(str(RAW_DATA_PATH)),
    )
    .withColumn(
        "_record_hash",
        record_hash_expression,
    )
)

In [14]:
bronze_record_count = bronze_df.count()

print(
    "Bronze record count:",
    f"{bronze_record_count:,}",
)

Bronze record count: 6,362,620


In [15]:
standardized_df = (
    bronze_df
    .withColumnRenamed(
        "type",
        "transaction_type",
    )
    .withColumnRenamed(
        "nameOrig",
        "origin_account",
    )
    .withColumnRenamed(
        "oldbalanceOrg",
        "old_balance_origin",
    )
    .withColumnRenamed(
        "newbalanceOrig",
        "new_balance_origin",
    )
    .withColumnRenamed(
        "nameDest",
        "destination_account",
    )
    .withColumnRenamed(
        "oldbalanceDest",
        "old_balance_destination",
    )
    .withColumnRenamed(
        "newbalanceDest",
        "new_balance_destination",
    )
    .withColumnRenamed(
        "isFraud",
        "is_fraud",
    )
    .withColumnRenamed(
        "isFlaggedFraud",
        "is_flagged_fraud",
    )
)

In [16]:
standardized_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- origin_account: string (nullable = true)
 |-- old_balance_origin: double (nullable = true)
 |-- new_balance_origin: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- old_balance_destination: double (nullable = true)
 |-- new_balance_destination: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- is_flagged_fraud: integer (nullable = true)
 |-- _pipeline_run_id: string (nullable = false)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _ingestion_date: date (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _source_file_path: string (nullable = false)
 |-- _record_hash: string (nullable = true)



In [17]:
standardized_df = (
    standardized_df
    .withColumn(
        "transaction_type",
        F.upper(F.trim(F.col("transaction_type"))),
    )
    .withColumn(
        "origin_account",
        F.trim(F.col("origin_account")),
    )
    .withColumn(
        "destination_account",
        F.trim(F.col("destination_account")),
    )
)

In [18]:
silver_enriched_df = (
    standardized_df
    .withColumn(
        "transaction_day",
        F.ceil(
            F.col("step") / F.lit(24)
        ).cast("integer"),
    )
    .withColumn(
        "hour_of_day",
        (
            (F.col("step") - F.lit(1))
            % F.lit(24)
        ).cast("integer"),
    )
)

In [19]:
silver_enriched_df.select(
    F.min("transaction_day").alias("minimum_day"),
    F.max("transaction_day").alias("maximum_day"),
    F.min("hour_of_day").alias("minimum_hour"),
    F.max("hour_of_day").alias("maximum_hour"),
).show()

+-----------+-----------+------------+------------+
|minimum_day|maximum_day|minimum_hour|maximum_hour|
+-----------+-----------+------------+------------+
|          1|         31|           0|          23|
+-----------+-----------+------------+------------+



In [20]:
silver_enriched_df = (
    silver_enriched_df
    .withColumn(
        "destination_category",
        F.when(
            F.col("destination_account").startswith("M"),
            F.lit("MERCHANT"),
        )
        .when(
            F.col("destination_account").startswith("C"),
            F.lit("CUSTOMER"),
        )
        .otherwise(
            F.lit("UNKNOWN")
        ),
    )
    .withColumn(
        "is_customer_destination",
        (
            F.col("destination_category")
            == F.lit("CUSTOMER")
        ).cast("integer"),
    )
    .withColumn(
        "is_merchant_destination",
        (
            F.col("destination_category")
            == F.lit("MERCHANT")
        ).cast("integer"),
    )
)

In [21]:
silver_enriched_df.groupBy(
    "destination_category"
).count().orderBy(
    F.desc("count")
).show()

+--------------------+-------+
|destination_category|  count|
+--------------------+-------+
|            CUSTOMER|4211125|
|            MERCHANT|2151495|
+--------------------+-------+



In [22]:
HIGH_VALUE_THRESHOLD = 200_000.0

silver_enriched_df = (
    silver_enriched_df
    .withColumn(
        "is_high_value_transaction",
        (
            F.col("amount")
            > F.lit(HIGH_VALUE_THRESHOLD)
        ).cast("integer"),
    )
    .withColumn(
        "is_zero_amount",
        (
            F.col("amount") == F.lit(0)
        ).cast("integer"),
    )
    .withColumn(
        "is_self_transfer",
        (
            F.col("origin_account")
            == F.col("destination_account")
        ).cast("integer"),
    )
)

In [23]:
silver_enriched_df = (
    silver_enriched_df
    .withColumn(
        "amount_category",
        F.when(
            F.col("amount") == 0,
            F.lit("ZERO"),
        )
        .when(
            F.col("amount") < 1_000,
            F.lit("LOW"),
        )
        .when(
            F.col("amount") < 50_000,
            F.lit("MEDIUM"),
        )
        .when(
            F.col("amount") <= 200_000,
            F.lit("HIGH"),
        )
        .otherwise(
            F.lit("VERY_HIGH"),
        ),
    )
)

In [24]:
silver_enriched_df.groupBy(
    "amount_category"
).count().orderBy(
    F.desc("count")
).show()

+---------------+-------+
|amount_category|  count|
+---------------+-------+
|         MEDIUM|2663305|
|           HIGH|1883103|
|      VERY_HIGH|1673570|
|            LOW| 142626|
|           ZERO|     16|
+---------------+-------+



In [25]:
silver_enriched_df = (
    silver_enriched_df
    .withColumn(
        "destination_balance_available",
        F.when(
            F.col("destination_category") == "MERCHANT",
            F.lit(0),
        ).otherwise(F.lit(1)),
    )
)

In [26]:
VALID_TRANSACTION_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

In [27]:
validated_df = (
    silver_enriched_df
    .withColumn(
        "fail_missing_critical_field",
        (
            F.col("step").isNull()
            | F.col("transaction_type").isNull()
            | F.col("amount").isNull()
            | F.col("origin_account").isNull()
            | F.col("destination_account").isNull()
            | F.col("is_fraud").isNull()
            | F.col("is_flagged_fraud").isNull()
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_transaction_type",
        (
            ~F.col("transaction_type").isin(
                VALID_TRANSACTION_TYPES
            )
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_step",
        (
            ~F.col("step").between(1, 744)
        ).cast("integer"),
    )
    .withColumn(
        "fail_negative_amount",
        (
            F.col("amount") < 0
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_fraud_flag",
        (
            ~F.col("is_fraud").isin(0, 1)
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_flagged_fraud",
        (
            ~F.col("is_flagged_fraud").isin(0, 1)
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_origin_account",
        (
            ~F.col("origin_account").rlike(
                r"^C[0-9]+$"
            )
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_destination_account",
        (
            ~F.col("destination_account").rlike(
                r"^[CM][0-9]+$"
            )
        ).cast("integer"),
    )
)

In [28]:
hard_failure_columns = [
    "fail_missing_critical_field",
    "fail_invalid_transaction_type",
    "fail_invalid_step",
    "fail_negative_amount",
    "fail_invalid_fraud_flag",
    "fail_invalid_flagged_fraud",
    "fail_invalid_origin_account",
    "fail_invalid_destination_account",
]

In [29]:
hard_failure_summary_df = validated_df.select(
    *[
        F.sum(F.col(column)).alias(column)
        for column in hard_failure_columns
    ]
)

hard_failure_summary_df.show(
    truncate=False
)

+---------------------------+-----------------------------+-----------------+--------------------+-----------------------+--------------------------+---------------------------+--------------------------------+
|fail_missing_critical_field|fail_invalid_transaction_type|fail_invalid_step|fail_negative_amount|fail_invalid_fraud_flag|fail_invalid_flagged_fraud|fail_invalid_origin_account|fail_invalid_destination_account|
+---------------------------+-----------------------------+-----------------+--------------------+-----------------------+--------------------------+---------------------------+--------------------------------+
|0                          |0                            |0                |0                   |0                      |0                         |0                          |0                               |
+---------------------------+-----------------------------+-----------------+--------------------+-----------------------+--------------------------+-------

In [30]:
validated_df = validated_df.withColumn(
    "hard_failure_count",
    sum(
        F.col(column)
        for column in hard_failure_columns
    ),
)

In [31]:
validated_df = (
    validated_df
    .withColumn(
        "origin_outgoing_balance_error",
        F.abs(
            F.col("old_balance_origin")
            - F.col("amount")
            - F.col("new_balance_origin")
        ),
    )
    .withColumn(
        "origin_cash_in_balance_error",
        F.abs(
            F.col("old_balance_origin")
            + F.col("amount")
            - F.col("new_balance_origin")
        ),
    )
)

In [32]:
BALANCE_TOLERANCE = 0.01

OUTGOING_TYPES = [
    "PAYMENT",
    "TRANSFER",
    "CASH_OUT",
    "DEBIT",
]

In [33]:
validated_df = (
    validated_df
    .withColumn(
        "warn_zero_amount",
        (
            F.col("amount") == 0
        ).cast("integer"),
    )
    .withColumn(
        "warn_self_transfer",
        (
            F.col("origin_account")
            == F.col("destination_account")
        ).cast("integer"),
    )
    .withColumn(
        "warn_origin_balance_mismatch",
        (
            F.col("transaction_type").isin(
                OUTGOING_TYPES
            )
            & (
                F.col("origin_outgoing_balance_error")
                > F.lit(BALANCE_TOLERANCE)
            )
        ).cast("integer"),
    )
    .withColumn(
        "warn_cash_in_balance_mismatch",
        (
            (F.col("transaction_type") == "CASH_IN")
            & (
                F.col("origin_cash_in_balance_error")
                > F.lit(BALANCE_TOLERANCE)
            )
        ).cast("integer"),
    )
    .withColumn(
        "warn_high_value_transfer_not_flagged",
        (
            (F.col("transaction_type") == "TRANSFER")
            & (
                F.col("amount")
                > F.lit(HIGH_VALUE_THRESHOLD)
            )
            & (F.col("is_flagged_fraud") == 0)
        ).cast("integer"),
    )
    .withColumn(
        "warn_flagged_transaction_rule_mismatch",
        (
            (F.col("is_flagged_fraud") == 1)
            & (
                (F.col("transaction_type") != "TRANSFER")
                | (
                    F.col("amount")
                    <= F.lit(HIGH_VALUE_THRESHOLD)
                )
            )
        ).cast("integer"),
    )
    .withColumn(
        "warn_merchant_balance_anomaly",
        (
            (F.col("destination_category") == "MERCHANT")
            & (
                (F.col("old_balance_destination") != 0)
                | (
                    F.col("new_balance_destination")
                    != 0
                )
            )
        ).cast("integer"),
    )
)

In [34]:
warning_columns = [
    "warn_zero_amount",
    "warn_self_transfer",
    "warn_origin_balance_mismatch",
    "warn_cash_in_balance_mismatch",
    "warn_high_value_transfer_not_flagged",
    "warn_flagged_transaction_rule_mismatch",
    "warn_merchant_balance_anomaly",
]

In [35]:
validated_df = validated_df.withColumn(
    "warning_count",
    sum(
        F.col(column)
        for column in warning_columns
    ),
)

In [36]:
warning_summary_df = validated_df.select(
    *[
        F.sum(F.col(column)).alias(column)
        for column in warning_columns
    ]
)

warning_summary_df.show(
    truncate=False
)

+----------------+------------------+----------------------------+-----------------------------+------------------------------------+--------------------------------------+-----------------------------+
|warn_zero_amount|warn_self_transfer|warn_origin_balance_mismatch|warn_cash_in_balance_mismatch|warn_high_value_transfer_not_flagged|warn_flagged_transaction_rule_mismatch|warn_merchant_balance_anomaly|
+----------------+------------------+----------------------------+-----------------------------+------------------------------------+--------------------------------------+-----------------------------+
|16              |0                 |3678407                     |101096                       |409094                              |0                                     |0                            |
+----------------+------------------+----------------------------+-----------------------------+------------------------------------+--------------------------------------+----------------

In [37]:
validated_df = validated_df.withColumn(
    "hard_failure_reasons",
    F.array_compact(
        F.array(
            F.when(
                F.col("fail_missing_critical_field") == 1,
                F.lit("MISSING_CRITICAL_FIELD"),
            ),
            F.when(
                F.col("fail_invalid_transaction_type") == 1,
                F.lit("INVALID_TRANSACTION_TYPE"),
            ),
            F.when(
                F.col("fail_invalid_step") == 1,
                F.lit("INVALID_STEP"),
            ),
            F.when(
                F.col("fail_negative_amount") == 1,
                F.lit("NEGATIVE_AMOUNT"),
            ),
            F.when(
                F.col("fail_invalid_fraud_flag") == 1,
                F.lit("INVALID_FRAUD_FLAG"),
            ),
            F.when(
                F.col("fail_invalid_flagged_fraud") == 1,
                F.lit("INVALID_FLAGGED_FRAUD"),
            ),
            F.when(
                F.col("fail_invalid_origin_account") == 1,
                F.lit("INVALID_ORIGIN_ACCOUNT"),
            ),
            F.when(
                F.col("fail_invalid_destination_account") == 1,
                F.lit("INVALID_DESTINATION_ACCOUNT"),
            ),
        )
    ),
)

In [38]:
validated_df = validated_df.withColumn(
    "warning_reasons",
    F.array_compact(
        F.array(
            F.when(
                F.col("warn_zero_amount") == 1,
                F.lit("ZERO_AMOUNT"),
            ),
            F.when(
                F.col("warn_self_transfer") == 1,
                F.lit("SELF_TRANSFER"),
            ),
            F.when(
                F.col("warn_origin_balance_mismatch") == 1,
                F.lit("ORIGIN_BALANCE_MISMATCH"),
            ),
            F.when(
                F.col("warn_cash_in_balance_mismatch") == 1,
                F.lit("CASH_IN_BALANCE_MISMATCH"),
            ),
            F.when(
                F.col(
                    "warn_high_value_transfer_not_flagged"
                ) == 1,
                F.lit("HIGH_VALUE_TRANSFER_NOT_FLAGGED"),
            ),
            F.when(
                F.col(
                    "warn_flagged_transaction_rule_mismatch"
                ) == 1,
                F.lit("FLAGGED_RULE_MISMATCH"),
            ),
            F.when(
                F.col("warn_merchant_balance_anomaly") == 1,
                F.lit("MERCHANT_BALANCE_ANOMALY"),
            ),
        )
    ),
)

In [39]:
validated_df = (
    validated_df
    .withColumn(
        "hard_failure_reason_text",
        F.concat_ws(
            "|",
            F.col("hard_failure_reasons"),
        ),
    )
    .withColumn(
        "warning_reason_text",
        F.concat_ws(
            "|",
            F.col("warning_reasons"),
        ),
    )
)

In [40]:
validated_df = validated_df.withColumn(
    "validation_status",
    F.when(
        F.col("hard_failure_count") > 0,
        F.lit("QUARANTINE"),
    )
    .when(
        F.col("warning_count") > 0,
        F.lit("VALID_WITH_WARNING"),
    )
    .otherwise(
        F.lit("VALID"),
    ),
)

In [41]:
validated_df.groupBy(
    "validation_status"
).count().orderBy(
    F.desc("count")
).show()

+------------------+-------+
| validation_status|  count|
+------------------+-------+
|VALID_WITH_WARNING|3787016|
|             VALID|2575604|
+------------------+-------+



In [42]:
quarantine_df = validated_df.filter(
    F.col("hard_failure_count") > 0
)

In [43]:
silver_valid_df = validated_df.filter(
    F.col("hard_failure_count") == 0
)

In [44]:
silver_record_count = silver_valid_df.count()
quarantine_record_count = quarantine_df.count()

print(
    "Bronze records:",
    f"{bronze_record_count:,}",
)

print(
    "Silver valid records:",
    f"{silver_record_count:,}",
)

print(
    "Quarantine records:",
    f"{quarantine_record_count:,}",
)

Bronze records: 6,362,620
Silver valid records: 6,362,620
Quarantine records: 0


In [45]:
count_reconciliation_passed = (
    bronze_record_count
    == silver_record_count
    + quarantine_record_count
)

print(
    "Count reconciliation passed:",
    count_reconciliation_passed,
)

Count reconciliation passed: True


In [46]:
assert count_reconciliation_passed, (
    "Bronze count does not equal "
    "Silver count plus quarantine count."
)

In [47]:
FINAL_SILVER_COLUMNS = [
    "_record_hash",
    "step",
    "transaction_day",
    "hour_of_day",
    "transaction_type",
    "amount",
    "amount_category",
    "origin_account",
    "destination_account",
    "destination_category",
    "is_customer_destination",
    "is_merchant_destination",
    "old_balance_origin",
    "new_balance_origin",
    "old_balance_destination",
    "new_balance_destination",
    "destination_balance_available",
    "is_high_value_transaction",
    "is_zero_amount",
    "is_self_transfer",
    "is_fraud",
    "is_flagged_fraud",
    "validation_status",
    "warning_count",
    "warning_reason_text",
    "_pipeline_run_id",
    "_ingestion_timestamp",
    "_ingestion_date",
    "_source_file",
    "_source_file_path",
]

In [48]:
silver_final_df = silver_valid_df.select(
    *FINAL_SILVER_COLUMNS
)

In [49]:
FINAL_QUARANTINE_COLUMNS = [
    "_record_hash",
    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "destination_account",
    "is_fraud",
    "is_flagged_fraud",
    "validation_status",
    "hard_failure_count",
    "hard_failure_reason_text",
    "warning_count",
    "warning_reason_text",
    "_pipeline_run_id",
    "_ingestion_timestamp",
    "_ingestion_date",
    "_source_file",
    "_source_file_path",
]

In [50]:
quarantine_final_df = quarantine_df.select(
    *FINAL_QUARANTINE_COLUMNS
)

In [51]:
silver_final_df.printSchema()

root
 |-- _record_hash: string (nullable = true)
 |-- step: integer (nullable = true)
 |-- transaction_day: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- amount_category: string (nullable = false)
 |-- origin_account: string (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- destination_category: string (nullable = false)
 |-- is_customer_destination: integer (nullable = false)
 |-- is_merchant_destination: integer (nullable = false)
 |-- old_balance_origin: double (nullable = true)
 |-- new_balance_origin: double (nullable = true)
 |-- old_balance_destination: double (nullable = true)
 |-- new_balance_destination: double (nullable = true)
 |-- destination_balance_available: integer (nullable = false)
 |-- is_high_value_transaction: integer (nullable = true)
 |-- is_zero_amount: integer (nullable = true)
 |-- is_self_transfer: integer (nullable = tru

In [52]:
silver_final_df.select(
    "_record_hash",
    "step",
    "transaction_day",
    "hour_of_day",
    "transaction_type",
    "amount",
    "amount_category",
    "origin_account",
    "destination_account",
    "destination_category",
    "is_fraud",
    "validation_status",
    "warning_reason_text",
).show(
    n=10,
    truncate=False,
)

+----------------------------------------------------------------+----+---------------+-----------+----------------+--------+---------------+--------------+-------------------+--------------------+--------+------------------+-----------------------+
|_record_hash                                                    |step|transaction_day|hour_of_day|transaction_type|amount  |amount_category|origin_account|destination_account|destination_category|is_fraud|validation_status |warning_reason_text    |
+----------------------------------------------------------------+----+---------------+-----------+----------------+--------+---------------+--------------+-------------------+--------------------+--------+------------------+-----------------------+
|43363620118f0498b51bb08895f4b298fd12bbb53fbd594d066e2475e837717f|1   |1              |0          |PAYMENT         |9839.64 |MEDIUM         |C1231006815   |M1979787155        |MERCHANT            |0       |VALID             |                       |


In [53]:
silver_final_df.filter(
    F.col("validation_status")
    == "VALID_WITH_WARNING"
).select(
    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "destination_account",
    "warning_count",
    "warning_reason_text",
).show(
    n=20,
    truncate=False,
)

+----+----------------+---------+--------------+-------------------+-------------+-------------------------------------------------------+
|step|transaction_type|amount   |origin_account|destination_account|warning_count|warning_reason_text                                    |
+----+----------------+---------+--------------+-------------------+-------------+-------------------------------------------------------+
|1   |PAYMENT         |4024.36  |C1265012928   |M1176932104        |1            |ORIGIN_BALANCE_MISMATCH                                |
|1   |DEBIT           |9644.94  |C1900366749   |C997608398         |1            |ORIGIN_BALANCE_MISMATCH                                |
|1   |PAYMENT         |11633.76 |C1716932897   |M801569151         |1            |ORIGIN_BALANCE_MISMATCH                                |
|1   |CASH_OUT        |229133.94|C905080434    |C476402209         |1            |ORIGIN_BALANCE_MISMATCH                                |
|1   |PAYMENT         |1563

In [54]:
quarantine_final_df.show(
    n=20,
    truncate=False,
)

+------------+----+----------------+------+--------------+-------------------+--------+----------------+-----------------+------------------+------------------------+-------------+-------------------+----------------+--------------------+---------------+------------+-----------------+
|_record_hash|step|transaction_type|amount|origin_account|destination_account|is_fraud|is_flagged_fraud|validation_status|hard_failure_count|hard_failure_reason_text|warning_count|warning_reason_text|_pipeline_run_id|_ingestion_timestamp|_ingestion_date|_source_file|_source_file_path|
+------------+----+----------------+------+--------------+-------------------+--------+----------------+-----------------+------------------+------------------------+-------------+-------------------+----------------+--------------------+---------------+------------+-----------------+
+------------+----+----------------+------+--------------+-------------------+--------+----------------+-----------------+------------------+-

In [55]:
silver_metrics_df = silver_final_df.agg(
    F.count("*").alias("silver_record_count"),
    F.countDistinct("_record_hash").alias(
        "distinct_record_hash_count"
    ),
    F.countDistinct("origin_account").alias(
        "distinct_origin_accounts"
    ),
    F.countDistinct("destination_account").alias(
        "distinct_destination_accounts"
    ),
    F.sum("is_fraud").alias("fraud_count"),
    F.sum("is_flagged_fraud").alias(
        "flagged_fraud_count"
    ),
    F.sum("is_high_value_transaction").alias(
        "high_value_transaction_count"
    ),
    F.sum("is_merchant_destination").alias(
        "merchant_destination_count"
    ),
    F.sum("is_customer_destination").alias(
        "customer_destination_count"
    ),
)

silver_metrics_df.show(
    truncate=False
)

+-------------------+--------------------------+------------------------+-----------------------------+-----------+-------------------+----------------------------+--------------------------+--------------------------+
|silver_record_count|distinct_record_hash_count|distinct_origin_accounts|distinct_destination_accounts|fraud_count|flagged_fraud_count|high_value_transaction_count|merchant_destination_count|customer_destination_count|
+-------------------+--------------------------+------------------------+-----------------------------+-----------+-------------------+----------------------------+--------------------------+--------------------------+
|6362620            |6362620                   |6353307                 |2722362                      |8213       |16                 |1673570                     |2151495                   |4211125                   |
+-------------------+--------------------------+------------------------+-----------------------------+-----------+---------

In [56]:
validation_status_summary_df = (
    validated_df
    .groupBy("validation_status")
    .agg(
        F.count("*").alias("record_count"),
        F.sum("is_fraud").alias("fraud_count"),
        F.round(
            F.sum("amount"),
            2,
        ).alias("total_amount"),
    )
    .orderBy(
        F.desc("record_count")
    )
)

validation_status_summary_df.show(
    truncate=False
)

+------------------+------------+-----------+------------------+
|validation_status |record_count|fraud_count|total_amount      |
+------------------+------------+-----------+------------------+
|VALID_WITH_WARNING|3787016     |2781       |8.8528623589504E11|
|VALID             |2575604     |5432       |2.5910670886473E11|
+------------------+------------+-----------+------------------+



In [57]:
warning_metric_expressions = [
    F.sum(F.col(column)).alias(column)
    for column in warning_columns
]

warning_metrics_row = validated_df.select(
    *warning_metric_expressions
).first()

In [58]:
warning_summary_pdf = pd.DataFrame(
    [
        {
            "warning_name": column,
            "record_count": int(
                warning_metrics_row[column]
            ),
        }
        for column in warning_columns
    ]
)

warning_summary_pdf

,warning_name,record_count
0,warn_zero_amount,16
1,warn_self_transfer,0
2,warn_origin_balance_mismatch,3678407
3,warn_cash_in_balance_mismatch,101096
4,warn_high_value_transfer_not_flagged,409094
5,warn_flagged_transaction_rule_mismatch,0
6,warn_merchant_balance_anomaly,0


In [59]:
hard_failure_metrics_row = validated_df.select(
    *[
        F.sum(F.col(column)).alias(column)
        for column in hard_failure_columns
    ]
).first()

In [60]:
hard_failure_summary_pdf = pd.DataFrame(
    [
        {
            "failure_name": column,
            "record_count": int(
                hard_failure_metrics_row[column]
            ),
        }
        for column in hard_failure_columns
    ]
)

hard_failure_summary_pdf

,failure_name,record_count
0,fail_missing_critical_field,0
1,fail_invalid_transaction_type,0
2,fail_invalid_step,0
3,fail_negative_amount,0
4,fail_invalid_fraud_flag,0
5,fail_invalid_flagged_fraud,0
6,fail_invalid_origin_account,0
7,fail_invalid_destination_account,0


In [61]:
pipeline_end_timestamp = datetime.now(
    timezone.utc
)

duration_seconds = (
    pipeline_end_timestamp
    - pipeline_start_timestamp
).total_seconds()

In [62]:
audit_record = {
    "pipeline_run_id": pipeline_run_id,
    "pipeline_name": "paysim_bronze_to_silver",
    "source_file": source_filename,
    "pipeline_start_timestamp": (
        pipeline_start_timestamp.isoformat()
    ),
    "pipeline_end_timestamp": (
        pipeline_end_timestamp.isoformat()
    ),
    "duration_seconds": duration_seconds,
    "bronze_record_count": bronze_record_count,
    "silver_record_count": silver_record_count,
    "quarantine_record_count": quarantine_record_count,
    "count_reconciliation_passed": (
        count_reconciliation_passed
    ),
    "pipeline_status": (
        "SUCCESS"
        if count_reconciliation_passed
        else "FAILED"
    ),
}

In [63]:
audit_pdf = pd.DataFrame(
    [audit_record]
)

audit_pdf.T

,0
pipeline_run_id,25e09ba1-b996-4bef-9d75-2b409e67afdd
pipeline_name,paysim_bronze_to_silver
source_file,PS_20174392719_1491204439457_log.csv
pipeline_start_timestamp,2026-07-24T18:44:16.915965+00:00
pipeline_end_timestamp,2026-07-24T18:55:16.190250+00:00
duration_seconds,659.274285
bronze_record_count,6362620
silver_record_count,6362620
quarantine_record_count,0
count_reconciliation_passed,True


In [64]:
audit_file_path = (
    AUDIT_OUTPUT_PATH
    / f"bronze_to_silver_audit_{pipeline_run_id}.csv"
)

warning_file_path = (
    QUALITY_OUTPUT_PATH
    / f"silver_warning_summary_{pipeline_run_id}.csv"
)

failure_file_path = (
    QUALITY_OUTPUT_PATH
    / f"silver_hard_failure_summary_{pipeline_run_id}.csv"
)

In [65]:
audit_pdf.to_csv(
    audit_file_path,
    index=False,
)

warning_summary_pdf.to_csv(
    warning_file_path,
    index=False,
)

hard_failure_summary_pdf.to_csv(
    failure_file_path,
    index=False,
)

In [66]:
print("Audit file:", audit_file_path)
print("Warning summary:", warning_file_path)
print("Failure summary:", failure_file_path)

Audit file: C:\Projects\paysim-financial-data-pipeline\data\gold\pipeline_audit\bronze_to_silver_audit_25e09ba1-b996-4bef-9d75-2b409e67afdd.csv
Warning summary: C:\Projects\paysim-financial-data-pipeline\data\gold\silver_quality\silver_warning_summary_25e09ba1-b996-4bef-9d75-2b409e67afdd.csv
Failure summary: C:\Projects\paysim-financial-data-pipeline\data\gold\silver_quality\silver_hard_failure_summary_25e09ba1-b996-4bef-9d75-2b409e67afdd.csv


In [67]:
silver_final_df.createOrReplaceTempView(
    "silver_paysim_transactions"
)

quarantine_final_df.createOrReplaceTempView(
    "quarantine_paysim_transactions"
)

In [68]:
spark.sql(
    """
    SELECT
        transaction_type,
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_amount,
        SUM(is_fraud) AS fraud_count,
        ROUND(
            100.0 * SUM(is_fraud) / COUNT(*),
            6
        ) AS fraud_rate_pct
    FROM silver_paysim_transactions
    GROUP BY transaction_type
    ORDER BY transaction_count DESC
    """
).show(
    truncate=False
)

+----------------+-----------------+------------------+-----------+--------------+
|transaction_type|transaction_count|total_amount      |fraud_count|fraud_rate_pct|
+----------------+-----------------+------------------+-----------+--------------+
|CASH_OUT        |2237500          |3.9441299522449E11|4116       |0.183955      |
|PAYMENT         |2151495          |2.809337113837E10 |0          |0.000000      |
|CASH_IN         |1399284          |2.3636739191246E11|0          |0.000000      |
|TRANSFER        |532909           |4.8529198726317E11|4097       |0.768799      |
|DEBIT           |41432            |2.2719922128E8    |0          |0.000000      |
+----------------+-----------------+------------------+-----------+--------------+



In [69]:
spark.sql(
    """
    SELECT
        validation_status,
        COUNT(*) AS record_count,
        SUM(is_fraud) AS fraud_count
    FROM silver_paysim_transactions
    GROUP BY validation_status
    ORDER BY record_count DESC
    """
).show(
    truncate=False
)

+------------------+------------+-----------+
|validation_status |record_count|fraud_count|
+------------------+------------+-----------+
|VALID_WITH_WARNING|3787016     |2781       |
|VALID             |2575604     |5432       |
+------------------+------------+-----------+



## Final Findings

### Standardization

- Source column names were converted into clear snake_case business names.
- Transaction types and account identifiers were trimmed and standardized.
- Source lineage metadata from the Bronze layer was preserved.

### Derived fields

- `transaction_day` and `hour_of_day` were derived from the hourly PaySim step.
- Destination accounts were classified as customers or merchants.
- High-value, zero-value, and self-transfer indicators were created.
- Transaction amounts were classified into business-friendly categories.

### Validation

- Hard validation failures were identified using structural and domain rules.
- Records with hard failures were routed to the quarantine dataset.
- Soft warnings were retained in Silver with reason codes.
- Balance inconsistencies were treated as warnings rather than automatic
  rejection rules.

### Reconciliation

- Bronze record count was reconciled against Silver plus quarantine.
- No source records were silently discarded.
- Pipeline audit and quality-summary files were generated.

### Silver-layer role

The Silver layer is now suitable for:

- business aggregations;
- fraud-monitoring summaries;
- customer transaction analysis;
- time-based analytics;
- downstream feature engineering.

## Engineering Decisions

1. Hard failures and soft warnings are handled separately.

2. Hard failures are quarantined because they violate essential structural or
   domain requirements.

3. Soft warnings are retained because unusual financial records may still be
   valid and should remain available for investigation.

4. Balance columns are retained in Silver for audit and analysis.

5. Balance columns will be excluded from the fraud-model feature table because
   the dataset documentation warns that they may introduce leakage.

6. Record hashes and ingestion metadata are preserved to maintain lineage from
   Raw through Bronze and Silver.

7. No full-data conversion to Pandas is performed. Pandas is used only for
   small audit and quality-summary outputs.

8. Physical Spark persistence remains deferred until the project is moved to
   WSL2 or Docker.

In [70]:
silver_final_df
quarantine_final_df
validated_df

DataFrame[step: int, transaction_type: string, amount: double, origin_account: string, old_balance_origin: double, new_balance_origin: double, destination_account: string, old_balance_destination: double, new_balance_destination: double, is_fraud: int, is_flagged_fraud: int, _pipeline_run_id: string, _ingestion_timestamp: timestamp, _ingestion_date: date, _source_file: string, _source_file_path: string, _record_hash: string, transaction_day: int, hour_of_day: int, destination_category: string, is_customer_destination: int, is_merchant_destination: int, is_high_value_transaction: int, is_zero_amount: int, is_self_transfer: int, amount_category: string, destination_balance_available: int, fail_missing_critical_field: int, fail_invalid_transaction_type: int, fail_invalid_step: int, fail_negative_amount: int, fail_invalid_fraud_flag: int, fail_invalid_flagged_fraud: int, fail_invalid_origin_account: int, fail_invalid_destination_account: int, hard_failure_count: int, origin_outgoing_balanc

In [71]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
